# Layer 5: PySpark Batch Processing & Feature Engineering

In this notebook, we transform raw JSON events into structured ML features. We extract `view_count`, `cart_count`, and `session_duration_seconds` using PySpark DataFrames.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, max as _max, min as _min, when, unix_timestamp

spark = SparkSession.builder.appName("FeatureEngineering").getOrCreate()
df = spark.read.format("delta").load("s3a://ecommerce-lakehouse/events_raw/")

# Drop invalid sessions
df = df.filter(col("user_session").isNotNull() & (col("user_session") != ""))
df = df.withColumn("event_time_sec", unix_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss 'UTC'"))

# Feature Engineering
features_df = df.groupBy("user_session").agg(
    _sum(when(col("event_type") == "view", 1).otherwise(0)).alias("view_count"),
    _sum(when(col("event_type") == "cart", 1).otherwise(0)).alias("cart_count"),
    _sum(col("price")).alias("total_session_value"),
    (_max(col("event_time_sec")) - _min(col("event_time_sec"))).alias("session_duration_seconds"),
    _max(when(col("event_type") == "purchase", 1).otherwise(0)).alias("label")
).fillna(0)

display(features_df.limit(5))